# Q5. Alternative Approach Implementation (Individual Component)

This notebook demonstrates an alternative tokenization technique using **Hugging Face's `BasicTokenizer`** and compares it with the group's approach (NLTK, Split, and Regex).

---
- ### *Siew Sheng Yao (TP068174)*

### 1. Implement Tokenization Using an Alternative Approach (3 marks)

The `BasicTokenizer` from Hugging Face's `transformers` library performs tokenization based on whitespace splitting, punctuation splitting, and Unicode-aware processing. It:

- Splits text on whitespace and punctuation characters
- Optionally lowercases the input text
- Handles Unicode categories (e.g., accents, Chinese characters) properly
- Strips accents from characters when configured

Unlike simple splitting methods, `BasicTokenizer` is designed as the first stage of BERT's tokenization pipeline, focusing on linguistically-aware pre-tokenization before subword segmentation.

In [2]:
from transformers import BasicTokenizer

# Load the data
file_path = '../assets/dataset-a/Data_1.txt'
with open(file_path, 'r') as file:
    text = file.read().strip()

print("Original Text:")
print(text)

Original Text:
Classification is the task of choosing the correct class label for a given input. In basic
classification tasks, each input is considered in isolation from all other inputs, and the set of labels is defined in advance. The basic classification task has a number of interesting variants. For example, in multiclass classification, each instance may be assigned multiple labels; in open-class classification, the set of labels is not defined in advance; and in sequence classification, a list of inputs are jointly classified.


In [3]:
# Initialize BasicTokenizer with lowercasing enabled (default behaviour)
tokenizer = BasicTokenizer(do_lower_case=True)

# Tokenize the text
tokens_basic = tokenizer.tokenize(text)

print("Tokenization using Hugging Face BasicTokenizer (do_lower_case=True):")
print(tokens_basic[:20], "...")
print(f"\nTotal tokens found: {len(tokens_basic)}")

Tokenization using Hugging Face BasicTokenizer (do_lower_case=True):
['classification', 'is', 'the', 'task', 'of', 'choosing', 'the', 'correct', 'class', 'label', 'for', 'a', 'given', 'input', '.', 'in', 'basic', 'classification', 'tasks', ','] ...

Total tokens found: 96


In [4]:
# Initialize BasicTokenizer without lowercasing for comparison
tokenizer_no_lower = BasicTokenizer(do_lower_case=False)

# Tokenize the text
tokens_no_lower = tokenizer_no_lower.tokenize(text)

print("Tokenization using Hugging Face BasicTokenizer (do_lower_case=False):")
print(tokens_no_lower[:20], "...")
print(f"\nTotal tokens found: {len(tokens_no_lower)}")

Tokenization using Hugging Face BasicTokenizer (do_lower_case=False):
['Classification', 'is', 'the', 'task', 'of', 'choosing', 'the', 'correct', 'class', 'label', 'for', 'a', 'given', 'input', '.', 'In', 'basic', 'classification', 'tasks', ','] ...

Total tokens found: 96


### 2. Compare and Contrast with the Group's Approach (2 marks)

| Feature | NLTK `word_tokenize` (Group) | `str.split()` (Group) | Regex `\\w+` (Group) | Hugging Face `BasicTokenizer` (Alternative) |
| :--- | :--- | :--- | :--- | :--- |
| Output Type | List of word/punctuation tokens | List of whitespace-split strings | List of alphanumeric tokens | List of lowercased word/punctuation tokens |
| Punctuation | Retained as separate tokens | Attached to adjacent words | Removed entirely | Split into separate tokens |
| Case Handling | Preserved as-is | Preserved as-is | Preserved as-is | Lowercased by default (`do_lower_case=True`) |
| Unicode Support | Basic support | No special handling | Matches `\\w` (locale-dependent) | Full Unicode-aware processing (CJK, accents) |
| Design Purpose | General-purpose linguistic tokenization | Basic manual text splitting | Pattern-based extraction | Pre-tokenization stage for transformer models |
| Dependencies | NLTK library + Punkt data download | None (built-in) | `re` module (built-in) | Hugging Face `transformers` library |

**Contrast:**

The group's primary approach (NLTK `word_tokenize`) uses linguistically-motivated rules from the Penn Treebank Tokenizer to split text. The `BasicTokenizer`, on the other hand, originates from the BERT model's tokenization pipeline. While both handle punctuation similarly by separating it into individual tokens, the key differences are: (1) `BasicTokenizer` automatically lowercases text by default, which the group approach does not, and (2) `BasicTokenizer` includes advanced Unicode handling such as accent stripping and CJK character isolation, which none of the group methods perform.

### 3. Explain Why the Alternative Approach is Better, Worse, or Just Different (5 marks)

**Evaluation:**

**Why It Is Different:**

The Hugging Face `BasicTokenizer` is "just different" in its design philosophy. It was created as the first-stage tokenizer for BERT and other transformer-based models, where the goal is to prepare text for subword tokenization (WordPiece). In contrast, the group's NLTK `word_tokenize` is designed as a standalone linguistic tokenizer following Penn Treebank conventions. Both split on whitespace and punctuation, but their downstream purposes diverge significantly.

**Pros (Better for specific tasks):**

1. **Built-in Lowercasing:** The `do_lower_case=True` option normalizes text automatically, eliminating the need for a separate `.lower()` call in the preprocessing pipeline. This reduces boilerplate code and the chance of forgetting the normalization step.
2. **Unicode and Multilingual Support:** `BasicTokenizer` handles accented characters (e.g., `café` → `cafe`), CJK characters (Chinese, Japanese, Korean), and other Unicode categories natively. The group's regex `\\w+` approach depends on locale settings, and `str.split()` has no such awareness at all.
3. **Transformer Pipeline Integration:** If the downstream task involves fine-tuning or using a pre-trained BERT-based model, using `BasicTokenizer` ensures consistency between preprocessing and model expectations. This avoids tokenization mismatches that can degrade model performance.
4. **Accent Stripping:** With `strip_accents=True`, it can normalize accented characters (e.g., `é` → `e`), which is useful for standardizing text from diverse sources.

**Cons (Worse for specific tasks):**

1. **Heavy Dependency:** The `transformers` library is a large dependency (~400MB+) compared to NLTK or Python built-in tools. For simple tokenization tasks, this is excessive overhead.
2. **Not a Complete Tokenizer:** `BasicTokenizer` is only the first stage of BERT's tokenization. On its own, it does not perform subword splitting (WordPiece), so it does not provide the full benefit of the transformer tokenization pipeline.
3. **Lossy Lowercasing by Default:** The default `do_lower_case=True` setting means case information is lost. For tasks like Named Entity Recognition (NER), where capitalization is an important feature (e.g., distinguishing "Classification" as a potential proper noun vs. a common noun), this can be harmful.
4. **Less Established for Academic NLP:** In traditional computational linguistics coursework and research, NLTK's tokenizers are the standard reference. Using `BasicTokenizer` outside of a transformer pipeline may introduce unfamiliarity without providing a clear benefit.

**Conclusion:**

The `BasicTokenizer` is best suited when the text analysis pipeline ultimately feeds into a BERT or transformer-based model, where consistent preprocessing is critical. For standalone text analytics tasks like those in Q1 (word frequency, stop word removal), the group's NLTK approach remains more practical due to its lighter dependency, established conventions, and case-preserving behaviour. The `BasicTokenizer` is therefore "just different" — optimized for a specific modern NLP workflow rather than being universally better or worse.